In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import openpyxl
import datetime

import warnings
warnings.filterwarnings("ignore")

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_excel("dataset/Customer-Purchase-History.xlsx")
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"])
df.head(5)

,CustomerID,Product,PurchaseDate,Quantity,UnitPrice,CustomerName,ProductCategory,PaymentMethod,ReviewRating,TotalPrice
0,C5361,Phone,2024-03-05,8,618.83,Customer C5361,Office Supplies,Cash,1,4950.64
1,C6231,Laptop,2025-06-21,7,366.22,Customer C6231,Electronics,Debit Card,3,2563.54
2,C7704,Chair,2023-06-25,5,634.51,Customer C7704,Office Supplies,Credit Card,4,3172.55
3,C2923,Printer,2023-09-30,3,508.63,Customer C2923,Office Supplies,Gift Card,1,1525.89
4,C4847,Monitor,2023-04-03,4,452.06,Customer C4847,Electronics,Credit Card,2,1808.24


In [23]:
df.columns

Index(['CustomerID', 'Product', 'PurchaseDate', 'Quantity', 'UnitPrice',
       'CustomerName', 'ProductCategory', 'PaymentMethod', 'ReviewRating',
       'TotalPrice'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CustomerID       1800 non-null   str           
 1   Product          1800 non-null   str           
 2   PurchaseDate     1800 non-null   datetime64[us]
 3   Quantity         1800 non-null   int64         
 4   UnitPrice        1800 non-null   float64       
 5   CustomerName     1800 non-null   str           
 6   ProductCategory  1800 non-null   str           
 7   PaymentMethod    1800 non-null   str           
 8   ReviewRating     1800 non-null   int64         
 9   TotalPrice       1800 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(5)
memory usage: 140.8 KB


In [14]:
df["Product"].value_counts()

Product
Laptop     280
Tablet     278
Desk       256
Monitor    254
Chair      253
Phone      246
Printer    233
Name: count, dtype: int64

In [49]:
df["ProductCategory"].value_counts()

ProductCategory
Electronics        606
Office Supplies    597
Furniture          597
Name: count, dtype: int64

# Sales Performance & Revenue

##### 1- Which product categories and individual products are the primary drivers of revenue, and how concentrated is revenue among top-performing products?

<p><b>Strategic Value:</b> Identifies the products/categories that deserve greater inventory, marketing, and management attention while revealing dependency on a small number of products.</p>

In [21]:
"""Calculate revenue per product category."""

product_category = df.groupby(["ProductCategory"],as_index=False)["TotalPrice"].sum()
product_category = product_category.sort_values(by="TotalPrice",ascending=False).reset_index(drop=True)

product_category.to_excel("export-data/Sales Performance & Revenue/Question 01/product_category_revenue.xlsx",index=False)

In [20]:
"""Calculate revenue by product category and products."""

product = df.groupby(["ProductCategory","Product"], as_index=False)["TotalPrice"].sum()
product = product.sort_values(by=["ProductCategory","TotalPrice"], ascending=[True,False]).reset_index(drop=True)

product.to_excel("export-data/Sales Performance & Revenue/Question 01/product_revenue.xlsx",index=False)

##### 2- How has sales revenue and sales volume evolved over time, and are there statistically meaningful periods of growth or decline?

<p><b>Strategic Value:</b> Reveals business growth patterns, seasonality, demand changes, and periods requiring investigation or intervention.</p>

In [18]:
"""Calculate yearly revenue to discover long-term growth."""

yearly_data = df.copy()
yearly_data["PurchaseDate(Year)"] = yearly_data["PurchaseDate"].dt.strftime("%Y")

yearly_revenue = yearly_data.groupby(["PurchaseDate(Year)"],as_index=False)["TotalPrice"].sum()

yearly_revenue.to_excel("export-data/Sales Performance & Revenue/Question 02/yearly_revenue.xlsx",index=False)

In [22]:
"""Calculate yearly revenue to identify trends, peaks and declines."""

monthly_data = df.copy()

monthly_data["PurchaseDate"] = pd.to_datetime(monthly_data["PurchaseDate"])
monthly_data["PurchaseDate(Year-Month)"] = monthly_data["PurchaseDate"].dt.strftime("%Y-%m")

monthly_revenue = monthly_data.groupby(["PurchaseDate(Year-Month)"],as_index=False)["TotalPrice"].sum()

monthly_revenue.to_excel("export-data/Sales Performance & Revenue/Question 02/monthly_revenue.xlsx",index=False)

##### 3- Which products generate high revenue but low sales volume, and which generate high volume but relatively low revenue?

<p><b>Strategic Value:</b> Helps distinguish premium/high-value products from volume-driven products and supports differentiated pricing and inventory strategies.</p>

In [23]:
revenue = df.groupby(["ProductCategory","Product"],as_index=False)[["Quantity","TotalPrice"]].sum()
revenue["ProductLabel"] = revenue["ProductCategory"] + " - " + revenue["Product"]
revenue["ASP"] = revenue["TotalPrice"] / revenue["Quantity"]
revenue = revenue.sort_values(by="TotalPrice",ascending=False).reset_index(drop=True)

revenue.to_excel("export-data/Sales Performance & Revenue/Question 03/revenue.xlsx",index=False)

# Customer Behavior & Value

##### 4- Who are the highest-value customers, and what proportion of total revenue comes from the top 10% of customers?

<p><b>Strategic Value:</b> Determines customer concentration and helps prioritize retention efforts around high-value accounts.</p>

In [55]:
high_value_customers = df.groupby(["CustomerName"],as_index=False).agg(
    Purchasefrequencey = ("CustomerName","count"),
    TotalRevenue = ("TotalPrice","sum"),
    Quantity = ("Quantity","sum"),
    AvgReviewRating = ("ReviewRating","mean")
    
)

high_value_customers=high_value_customers.sort_values(by="TotalRevenue",ascending=False).reset_index(drop=True)
high_value_customers.to_excel("export-data/Customer Behavior & Value/Question 01/high_value_customers.xlsx",index=False)


top_10 = int(len(high_value_customers) * 0.10)
top_10_revenue = high_value_customers.head(top_10)["TotalRevenue"].sum()
total_revenue = high_value_customers["TotalRevenue"].sum()
top_10_percent = (top_10_revenue / total_revenue) * 100

print("Number of top 10% of customers :",top_10)
print("Revenue of top 10% of customers :",top_10_revenue)
print("Total Revenue :",total_revenue)
print(f"Proportion of total revenue comes from the top 10% of customers is {top_10_percent:,.2f}%")

Number of top 10% of customers : 164
Revenue of top 10% of customers : 895244.7999999999
Total Revenue : 3266656.8
Proportion of total revenue comes from the top 10% of customers is 27.41%


##### 5- What customer segments can be identified based on purchase frequency, monetary value, and recency?

<p><b>Strategic Value:</b> Enables targeted marketing instead of treating every customer identically.</p>

##### 6- Do customers who purchase more frequently also generate disproportionately higher revenue?

<p><b>Strategic Value:</b> Tests whether increasing purchase frequency is associated with higher customer lifetime value and can inform loyalty/retention programs.</p>

##### 7- Which products or categories are associated with the highest and lowest customer satisfaction ratings?

<p><b>Strategic Value:</b> Connects commercial performance with customer experience and can identify products requiring quality, pricing, or service improvements.</p>